# K2 evaluation (EXPLORATORY)
CPU fake-model default. Step 1 runs all three arms; step 2 requires a Claude gate file bound to the downloaded step-1 manifest. Issuer batteries use L1 exact rewrites and Compliance Director stops. Values are exported for separately authorized Haiku grading; no grader call is made here.


In [ ]:
from pathlib import Path
import sys, json, hashlib, os
DRY_RUN = True
MODE = 'evaluate'
REPO = Path('/kaggle/input/taa-k2-repo/taa-lab')
# For local CPU dry-run only, set REPO to the checkout root.
PINNED_COMMIT = ''  # Full reviewed K2 commit; dataset metadata must match.
MODEL = 'gemma-3-1b-it'  # or gemma-3-4b-it
MODEL_PATH = Path('/kaggle/input/gemma-model/transformers/default/1')
MODEL_REVISION = ''  # Versioned Kaggle Models handle.
OUTPUT = Path('/kaggle/working/k2-output')
if not DRY_RUN and (len(PINNED_COMMIT)!=40):
    raise ValueError('Pin repo and model revisions before running')
if not DRY_RUN:
    assert (REPO/'COMMIT.txt').read_text().strip()==PINNED_COMMIT
sys.path.insert(0,str(REPO))
from exploratory.k2_kaggle import core
if not DRY_RUN:
    core.validate_model_source(MODEL_REVISION, MODEL)
OUTPUT.mkdir(parents=True,exist_ok=False)
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
# Kaggle T4 sessions have two GPUs; with both visible the Trainer uses DataParallel
# and silently doubles the batch. The recipe is one GPU, batch 1.
os.environ['CUDA_VISIBLE_DEVICES']='0'
metadata = dict(dry_run=DRY_RUN, repo_commit=PINNED_COMMIT, model_id=core.model_id(MODEL), model_revision=MODEL_REVISION,
                asset_hashes={})
def retain(row):
    with (OUTPUT/'events.jsonl').open('a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False)+'\n')
STEP=1
SHARD_ARM = None
SHARD_KIND = None
REPEATS=1
GATE_FILE=None
ADAPTERS={'neutral':Path('/kaggle/input/k2-neutral/adapter'),'contrary':Path('/kaggle/input/k2-contrary/adapter')}
metadata.update(step=STEP,repeats=REPEATS,sampling=dict(do_sample=True,temperature=1.0,top_p=1.0),grader_max_tokens=1024)

SESSION_HOURS=8.5
import time
DEADLINE=time.monotonic()+SESSION_HOURS*3600
metadata.update(mode=MODE,recipe_sha256=core.digest(core.RECIPE),implementation_sha256=hashlib.sha256(Path(core.__file__).read_bytes().replace(b'\r\n',b'\n')).hexdigest())
(OUTPUT/'metadata.json').write_text(json.dumps(metadata,indent=2))
def check_deadline(): core.deadline_check(DEADLINE)
REPEAT_START=1
TRAIN_SEED=247056
TRAINING_DIGESTS={}
metadata.update(training_seed=TRAIN_SEED, repeat_start=REPEAT_START)


In [ ]:
try:
    if STEP==1 and not DRY_RUN and SHARD_ARM is None and SHARD_KIND is None:
        raise ValueError('non-dry step 1 requires a shard')
    if SHARD_ARM is not None or SHARD_KIND is not None:
        selected=core.shard_cells(core.plan(STEP,REPEATS,MODEL,MODEL_REVISION,REPEAT_START,TRAIN_SEED),SHARD_ARM,SHARD_KIND)
        metadata['shard']=dict(arm=SHARD_ARM,kind=SHARD_KIND,cell_ids=[x['case_id'] for x in selected])
        ADAPTERS={a:p for a,p in ADAPTERS.items() if a==SHARD_ARM}
    metadata['asset_hashes']=core.validate_assets()
    gate=json.loads(Path(GATE_FILE).read_text()) if GATE_FILE else None
    if not DRY_RUN:
        check_deadline()
        metadata['wheels']=core.install_wheels()
        metadata['packages']=core.check_stack()
        check_deadline()
        MODEL_PATH=core.resolve_model_path(MODEL_PATH)
        metadata['base_inventory_sha256']=core.digest(core.inventory(MODEL_PATH))
        import torch, transformers, peft
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
        metadata['gpu'] = torch.cuda.get_device_name(0)
        if torch.cuda.device_count()!=1: raise RuntimeError('expected exactly one visible GPU')
        metadata['visible_gpus']=torch.cuda.device_count()
        check_deadline()
        tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH,local_files_only=True,trust_remote_code=False)
        check_deadline()
        model=core.load_model(MODEL,transformers,MODEL_PATH,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16),device_map={'':0})
        check_deadline()
        metadata['loaded_class']=type(model).__name__
        metadata['resolved_generation_config']=model.generation_config.to_dict()
        def prompt(messages): return core.prompt(tokenizer,messages)
    generate=core.fake_model
    if not DRY_RUN:
        from peft import PeftModel
        # Attach fresh K2 adapters only, trained against this exact snapshot.
        for arm,path in ADAPTERS.items():
            expected=dict(model_id=core.model_id(MODEL),model_revision=MODEL_REVISION,arm=arm,
                training_seed=TRAIN_SEED,asset_hashes=core.validate_assets(),recipe_sha256=core.digest(core.RECIPE),
                base_inventory_sha256=metadata['base_inventory_sha256'],repo_commit=PINNED_COMMIT,
                implementation_sha256=metadata['implementation_sha256'])
            check_deadline()
            TRAINING_DIGESTS[arm]=core.verify_adapter(path,expected)
            (OUTPUT/(arm+'-training-manifest.json')).write_bytes((path.parent/'manifest.json').read_bytes())
        metadata['training_manifest_digests']=TRAINING_DIGESTS
        check_deadline()
        for i,(arm,path) in enumerate(ADAPTERS.items()):
            check_deadline()
            if i==0: model=PeftModel.from_pretrained(model,path,adapter_name=arm,is_trainable=False)
            else: model.load_adapter(path,adapter_name=arm,is_trainable=False)
        model.eval()
        generation=model.generation_config.to_dict()
        generation.update(do_sample=True,temperature=1.0,top_p=1.0,
                eos_token_id=[tokenizer.eos_token_id,tokenizer.convert_tokens_to_ids('<end_of_turn>')],pad_token_id=tokenizer.pad_token_id)
        metadata['resolved_generation_config']=generation
        metadata['adapters']={arm:{str(p.relative_to(path)):hashlib.sha256(p.read_bytes()).hexdigest()
            for p in path.rglob('*') if p.is_file()} for arm,path in ADAPTERS.items()}
        def generate(system,history,cap,seed,arm):
            check_deadline()
            set_seed(seed)
            messages=core.evaluation_messages(system,history)
            ids=prompt(messages)
            if len(ids)+cap>16384: raise ValueError('Evaluation context exceeds 16384; no truncation')
            inputs=torch.tensor([ids],device='cuda:0')
            def complete():
                check_deadline()
                core.generation_started()  # issuance boundary: after every pre-generation guard
                with torch.no_grad():
                    result=model.generate(inputs,max_new_tokens=cap,do_sample=True,temperature=1.0,top_p=1.0,
                        eos_token_id=[tokenizer.eos_token_id,tokenizer.convert_tokens_to_ids('<end_of_turn>')],
                        pad_token_id=tokenizer.pad_token_id)
                raw_ids=result[0,len(ids):].tolist()
                retain(dict(event='native_tokens',seed=seed,max_new_tokens=cap,input_ids=ids,output_ids=raw_ids))
                return tokenizer.decode(raw_ids,skip_special_tokens=True)
            if arm=='base':
                if not ADAPTERS: return complete()
                with model.disable_adapter(): return complete()
            model.set_adapter(arm)
            return complete()
    if STEP==1 and not DRY_RUN:
        counts={arm:{} for arm in core.ARMS}
        for arm in core.acquisition_arms(SHARD_ARM,SHARD_KIND):
            for i,messages in enumerate(core.acquisition_prompts()):
                raw=generate(messages[0]['content'],messages[1:],128,248017+i,arm)
                label=core.acquisition_score(raw,messages)
                retain(dict(event='acquisition',arm=arm,index=i,raw=raw,label=label,**core.acquisition_result(raw,messages),distribution='in-distribution training records'))
                counts[arm][label]=counts[arm].get(label,0)+1
        (OUTPUT/'acquisition-counts.json').write_text(json.dumps(counts,indent=2))
    core.run(STEP,generate,retain,REPEATS,gate=gate,dry_run=DRY_RUN,model=MODEL,model_revision=MODEL_REVISION,
        repeat_start=REPEAT_START,training_seed=TRAIN_SEED,training_digests=TRAINING_DIGESTS,check_deadline=check_deadline,shard_arm=SHARD_ARM,shard_kind=SHARD_KIND)
except BaseException as exc:
    retain(dict(event='run_failed',error=type(exc).__name__,message=str(exc)))
    raise
finally:
    (OUTPUT/'metadata.json').write_text(json.dumps(metadata,indent=2))
    try:
        if STEP==1:
            rows=core.finalize(OUTPUT/'events.jsonl',DRY_RUN) if (OUTPUT/'events.jsonl').exists() else []
            planned=core.plan(1,REPEATS,MODEL,MODEL_REVISION,REPEAT_START,TRAIN_SEED,TRAINING_DIGESTS)
            summary=core.sanity_summary(rows,core.model_id(MODEL),MODEL_REVISION,planned)
            if 'shard' in metadata: summary['partial_shard']=True
            (OUTPUT/'sanity-summary.json').write_text(json.dumps(summary,indent=2))
    finally:
        core.manifest(OUTPUT,metadata)


## Values grading handoff
Export the 16 values responses per checkpoint and repeat under blinded identifiers. A separately approved Haiku run must use the existing `k1_values.RUBRIC`, strict JSON validation, and **max_tokens=1024** fixed before grading. Keep raw content blocks, stop reason, usage, invalid grades and refusals. Do not call `k1_values.score` directly: its historical cap is 256. No API token belongs in this notebook.
